# Loop0, stage 00: read the wells, label them, concatenate

The first and largest loop.  It pools every plate acquired up to that point, draws the gating strategy by hand, builds the fluorescence-minus-one calibration that every later loop reuses, and drops the experiments that were not part of the screen.

## Inputs

| file | produced by | holds |
|---|---|---|
| `experiment_table.csv` | this folder | maps an acquisition date and start time to an experiment number |
| `metadata.csv` | this folder | culture conditions, one row per experiment |
| `$LABCOMPASS_FCS_ROOT/` | acquisition, outside the repository | one unmixed `.fcs` file per well |

## Outputs

| file | holds |
|---|---|
| `Loop0.h5ad` | every cell of every well, labelled, with the antibody channels only |
| `Loop0_500k.h5ad` | about 500,000 of those cells, spread evenly over the experiments |

Run the cells from top to bottom.  Nothing below reads a file that a later cell
writes, so `Restart Kernel and Run All Cells` reproduces both outputs.

## Setup

Every path and every number that this notebook uses is set here.  This notebook
needs nothing from `labcompass.py`: every step is written out where it happens.

In [ ]:
import gc
import glob
import os

import anndata as ann
import numpy as np
import pandas as pd
import pytometry as pm

import sys

sys.path.append("..")  # labcompass.py sits beside the loop folders
from labcompass import downsample

# ---------------------------------------------------------------- configuration
# Where the unmixed .fcs files live.  They are far too large to commit, so the
# repository only records the path.  Override it without editing this notebook
# by setting the environment variable LABCOMPASS_FCS_ROOT.
FCS_ROOT = os.environ.get("LABCOMPASS_FCS_ROOT", '/rds/project/rds-SDzz0CATGms/unsorted/for_Juan_Licyel')
DATA_PATH = os.path.join(FCS_ROOT, '')
METADATA_CSV = 'metadata.csv'

OUT_ALL = "Loop0.h5ad"
OUT_SUBSET = "Loop0_500k.h5ad"

# Cap the downsampled file at this many cells in total, spread evenly over the
# experiments.  It exists so the gating notebook has something small to plot.
DESIRED_CELL_NUMBER = 500000
DOWNSAMPLE_SEED = 0

# The live/dead channel.  It is copied into .obs before it is dropped, because
# the live gate is drawn against it.
LIVE_DEAD_CHANNEL = "Live_Dead : 7-AAD - Area"

# Channels that carry no antibody signal.  The seven "[AF color N]" channels are
# the unmixed autofluorescence components the spectral cytometer reports.  CD184
# is dropped because its staining failed in this panel.
CHANNELS_TO_DROP = [
    "CD184 : PerCP-eFluor710 - Area",
    LIVE_DEAD_CHANNEL,
    "[AF color 1] - Area",
    "[AF color 2] - Area",
    "[AF color 3] - Area",
    "[AF color 4] - Area",
    "[AF color 5] - Area",
    "[AF color 6] - Area",
    "[AF color 7] - Area",
]

## The wells to read

Every unmixed `.fcs` file below `DATA_PATH` is read.
Only files inside a directory called `filtered` are taken, because that is
where the unmixing software writes its output.
The list is sorted, so the order does not depend on the filesystem.

In [ ]:
files = sorted(glob.glob(os.path.join(DATA_PATH, "**", "filtered", "*AutoSpectral.fcs"), recursive=True))
print(f"{len(files)} wells to read")
if not files:
    raise FileNotFoundError(f"no .fcs files found under {DATA_PATH}")

## The condition table

One row per experiment, holding the culture conditions.  The index is forced to
text.  Without that, pandas infers an integer index when every experiment number
happens to be numeric, and a later lookup with a string key silently finds
nothing.

In [ ]:
metadata = pd.read_csv(METADATA_CSV, sep="\t").set_index("experiment_number")
metadata.index = metadata.index.astype(str)
condition_columns = metadata.columns.tolist()

print(f"{len(metadata)} experiments, {len(condition_columns)} condition columns")
metadata.head()

## The acquisition table

One row per acquisition, holding the date and start time that identify the
`.fcs` file, the experiment number, and the bead and cell counts recorded by
hand.

In [ ]:
import csv

with open("experiment_table.csv", newline="") as handle:
    experiment_list = list(csv.DictReader(handle))

n_experiments = len({row["experiment_number"] for row in experiment_list})
print(f"{len(experiment_list)} acquisitions, {n_experiments} experiments")

## Read each well and label it

Each well is matched to an experiment by its acquisition date and start time.
A matched row is deleted from `experiment_list` so two wells cannot claim the
same acquisition.

A well whose acquisition is not in the table, or whose experiment number is not
in the condition table, is skipped and the reason is printed.  The acquisitions
left over at the end are printed too, so a mismatch is visible rather than
silent.

Four pieces of metadata come from the `.fcs` file itself, written there by the
cytometer: the acquisition date, the cytometer make, its serial number and the
well identifier.  The culture conditions come from the condition table.

`pytometry.pp.split_signal` moves the scatter and time channels from `.X` into
`.obs` and keeps only the area measurements in `.X`.

In [ ]:
adatas = []
skipped = []

for path in files:
    adata = pm.io.read_fcs(path)
    adata.obs.index = adata.obs.index.astype(str)
    # var_names has to hold the channel names, so a gate saved against a channel
    # can be applied before split_signal moves the scatter channels into .obs.
    adata.var_names = adata.var["marker"].astype(str)
    if adata.n_obs == 0:
        continue

    match = next(
        (
            i
            for i, row in enumerate(experiment_list)
            if row["date"] == adata.uns["meta"]["date"]
            and row["btim"] == adata.uns["meta"]["btim"]
        ),
        None,
    )
    if match is None:
        skipped.append((path, "acquisition not in experiment_table.csv"))
        continue

    row = experiment_list[match]
    experiment_number = row["experiment_number"]
    if experiment_number not in metadata.index:
        skipped.append((path, f"experiment {experiment_number} not in {METADATA_CSV}"))
        continue

    adata.obs["experiment_number"] = int(experiment_number)
    adata.obs["replicate"] = int(row["replicate"])
    adata.obs["count_beads"] = row["count_beads"]
    adata.obs["cell_counts"] = row["cell_counts"]
    adata.obs["date"] = adata.uns["meta"]["date"]
    adata.obs["cytometer"] = adata.uns["meta"]["cyt"]
    adata.obs["cytometer_serial_no"] = adata.uns["meta"]["cytsn"]
    adata.obs["well_id"] = adata.uns["meta"]["smno"]
    adata.obs[condition_columns] = metadata.loc[str(experiment_number)].tolist()

    pm.pp.split_signal(adata, var_key="channel", option="area", data_type="facs")
    adatas.append(adata)

    # Remove the matched acquisition so a second well cannot claim it.
    del experiment_list[match]

del adata
gc.collect()

print(f"{len(adatas)} wells kept, {len(skipped)} skipped")
for path, reason in skipped:
    print("  skipped", path, "|", reason)
print(f"{len(experiment_list)} acquisitions were never matched to a well")

## Concatenate

`join="inner"` keeps only the channels present in every well.
`index_unique="-"` appends each well's position to its cell names, so no two
cells share a name and later steps can index by name safely.

In [ ]:
adata_all = ann.concat(
    adatas,
    join="inner",
    merge="same",
    uns_merge="unique",
    label="batch",
    index_unique="-",
)
del adatas
gc.collect()
adata_all

## Move the live/dead channel, drop the channels we do not analyse, rename the rest

A channel name from the unmixing software has the form
`"<antibody> : <fluorochrome> - <parameter>"`, for example
`"CD34 : APC - Area"`.  It is split so `.var_names` holds the plain antibody
name and `.var["antibody"]` keeps it as a category.

In [ ]:
# The live gate is drawn against 7-AAD, and gating reads from .obs, so the
# channel has to be copied across before it is dropped from .X below.
values = adata_all[:, LIVE_DEAD_CHANNEL].X
adata_all.obs["7-AAD"] = np.asarray(
    values.toarray() if hasattr(values, "toarray") else values
).ravel()

keep = ~np.isin(adata_all.var["marker"].values, CHANNELS_TO_DROP)
adata_all = adata_all[:, keep]

markers = list(adata_all.var["marker"].values)
antibodies = [marker.split(" :")[0] for marker in markers]
fluorochromes = [marker.split(": ")[1].split(" -")[0] for marker in markers]

duplicates = sorted({a for a in antibodies if antibodies.count(a) > 1})
if duplicates:
    raise ValueError(f"channels reduce to duplicate antibody names: {duplicates}")

adata_all.var = adata_all.var.copy()
adata_all.var["antibody"] = pd.Categorical(antibodies)
adata_all.var_names = antibodies

print("antibodies:   ", antibodies)
print("fluorochromes:", fluorochromes)
adata_all

## Write the full object

This is the input of stage 02.

In [ ]:
adata_all.write(OUT_ALL)
print(f"wrote {OUT_ALL}: {adata_all.n_obs:,} cells x {adata_all.n_vars} antibodies")

## Write the downsampled object

Each experiment is capped at the same number of cells.  An experiment with fewer
cells than the cap is kept whole, so the total comes out below the target.

In [ ]:
target_size = int(DESIRED_CELL_NUMBER / adata_all.obs["experiment_number"].nunique())
print(f"cap of {target_size:,} cells per experiment")

adata_small = downsample(
    adata_all, obs="experiment_number", target_size=target_size, seed=DOWNSAMPLE_SEED
)
adata_small.write(OUT_SUBSET)
print(f"wrote {OUT_SUBSET}: {adata_small.n_obs:,} cells")